In [1]:
import sys
sys.path.append("../../")

import torch
import torchvision.utils
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.autograd import Variable
from torch import optim
from torch.utils.data import DataLoader
from torchvision import models
from torchsummary import summary

import numpy as np
import cv2
import pandas as pd
from sklearn.metrics import accuracy_score

import src.dataPreparation.CreatePartial as create_partial
import src.neuralNetworksArch.BstCnn as bst
import src.utils.Visual as vis
import src.utils.Checkpoint as ckp
import src.utils.Metrics as metrics

from src.config.Param import *

In [11]:
PROB_SAME = {
    'partial_1' : [
        0.3,
        0.5,
        0.2
    ],
    'partial_2' : [
        0.352697095,
        0.369294606,
        0.278008299
    ],
    'partial_3' : [
        0.26993865,
        0.251533742,
        0.260736196,
        0.217791411
    ]
}

PROB_DIFF = {
    'partial_1' : [
        0.336099585,
        0.356846473,
        0.307053942
    ],
    'partial_2' : [
        0.330396476,
        0.374449339,
        0.295154185
    ],
    'partial_3' : [
        0.266666667,
        0.25,
        0.26,
        0.223333333
    ]
}

In [25]:
MODEL_PATH = [
    '../../models/PARTIAL_1 #1.pth',
    '../../models/PARTIAL_1 #2.pth',
    '../../models/PARTIAL_1 #3.pth',
#     '../../models/PARTIAL_3 #4.pth'
]

PATH = '../../dataset/testing/same_cam/'
DATATEST_PATH = PATH + 'testing.csv'
IMAGES_PATH = PATH + '/images/full/'
IMAGES_OCCL_PATH = PATH + '/images/occl_20/'

In [26]:
def cv_image2tensor(img, transform, convert=True):
    if convert:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = transform(img)
    img = [img]
    img_tensors = torch.stack(img)

    return img_tensors

In [27]:
models = []
for index, path in enumerate(MODEL_PATH):
    data = {}
    model  = bst.BstCnn()
    checkpoint = ckp.load_checkpoint(load_dir=path)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    
    data['id'] = 'PART-' + str(index+1)
    data['model'] = model
    data['max_dist'] = checkpoint['max_dist']
    data['threshold'] = checkpoint['threshold']
    models.append(data)

In [28]:
trans = transforms.Compose([transforms.ToTensor()])
df = pd.read_csv(DATATEST_PATH)
y_true = []
y_pred = []

verbose = False
with torch.no_grad():
    for index, data in df.iterrows():
        if verbose == True:
            print('No-{}'.format(index+1))
            print('-'*50)
        img1 = cv2.imread(IMAGES_PATH + data['image_1'])
        img2 = cv2.imread(IMAGES_OCCL_PATH + data['image_2'])
        label = data['label']

        img1_part = list(create_partial.partial_image_1(img1))
        img2_part = list(create_partial.partial_image_1(img2))

        thresholds = []
        dists = []
        for i, (input1, input2, model) in enumerate(zip(img1_part, img2_part, models)):
            input1 = cv_image2tensor(img1, trans)
            input2 = cv_image2tensor(img2, trans)
            out1, out2 = model['model'](input1, input2)
            euclidean_distance = F.pairwise_distance(out1, out2)
            
            dist = metrics.normalize_dist(euclidean_distance.item(), model['max_dist'])
            dists.append(dist)
            thresholds.append(model['threshold'])
            
            if verbose == True:
                print('PART-{} DISTANCE => {}'.format((i+1), dist))
                print('Threshold => {}'.format(model['threshold']))
                print('-'*50)
            
        cat_dist, cat_thresh = metrics.concatenate(dists, thresholds, PROB_SAME['partial_1'])
        
        if verbose == True:
            print('actual distance => {}'.format(cat_dist))
            print('actual thresh => {}'.format(cat_thresh))
            print('actual label => {}'.format(label))
        
        y_true.append(float(label))
        pred = 0.0 if cat_dist <= cat_thresh else 1.0
        y_pred.append(pred)
        concatenated = torch.cat((cv_image2tensor(img1, trans, True), cv_image2tensor(img2, trans, True)),0)
        
        if verbose == True:
            vis.imshow(torchvision.utils.make_grid(concatenated))
            print('='*50, end='\n\n')

acc = accuracy_score(np.array(y_true), np.array(y_pred))
print('Accuracy : {}'.format(acc))

Accuracy : 0.8388888888888889
